## <font color="royalblue">**Estructuracion skills y requisitos (post-NLP)**</font>
Preparación de los datos para su visualización y análisis.
- Estructuración de skills globales y por ofertas.
- Estructuración de familias de skills.
- Estructuración de requisitos.
- Cruce de familias - requisitos.


In [52]:
import pandas as pd
import numpy as np
import ast

In [ ]:
# Cargar los DataFrames de las ofertas por portal
df_Adzuna = pd.read_csv("df_Adzuna_ofertas.csv")
df_Indeed = pd.read_csv("df_Indeed_ofertas.csv")
df_LinkedIn = pd.read_csv("df_LinkedIn_ofertas.csv")

# Cargar los DataFrames de los skills por globales por portal
df_indeed_skills_global = pd.read_csv("df_indeed_skills_global.csv")
df_linkedin_skills_global = pd.read_csv("df_linkedin_skills_global.csv")
df_adzuna_skills_global = pd.read_csv("df_adzuna_skills_global.csv")

In [3]:
df_LinkedIn = df_LinkedIn.rename(columns={"skills": "skill", "id": "id_oferta", "portal_web": "portal"})
df_Indeed = df_Indeed.rename(columns={"skills": "skill", "id": "id_oferta", "portal_web": "portal"})
df_Adzuna = df_Adzuna.rename(columns={"skills": "skill", "id": "id_oferta", "portal_web": "portal"})


In [4]:
# Funciones genéricas de estructuración
def generar_tablas(df_base, columna):
    conteo = (
        df_base
        .groupby(["portal", columna])
        .size()
        .reset_index(name="conteo")
    )
    tabla = conteo.pivot(index="portal", columns=columna, values="conteo").fillna(0)
    tabla_pct = tabla.div(tabla.sum(axis=1), axis=0)
    return tabla, tabla_pct

def generar_tablas_conteo(df_base, columna):
    tabla = df_base.pivot(index="portal", columns=columna, values="conteo").fillna(0)
    tabla_pct = tabla.div(tabla.sum(axis=1), axis=0)
    return tabla, tabla_pct



### Estructuración de Skills globales

In [5]:
df_indeed_skills_global["portal"] = "Indeed"
df_linkedin_skills_global["portal"] = "LinkedIn"
df_adzuna_skills_global["portal"] = "Adzuna"

df_skills_global = pd.concat([
    df_linkedin_skills_global, 
    df_indeed_skills_global, 
    df_adzuna_skills_global], ignore_index=True)

df_skills_global_conteo = (
    df_skills_global
    .groupby(["portal", "skill"])
    .size()
    .reset_index(name="conteo")
)


In [6]:
tab_skills_global, tab_skills_global_pct = generar_tablas(df_skills_global, "skill")
tab_skills_global

skill,adaptabilidad,airflow,alemán,angular,api,aws,azure,backend,bbdd,catalán,...,spark,sql,sql server,tableau,tensorflow,terraform,trabajo en equipo,typescript,warehouse,xgboost
portal,,,,,,,,,,,,,,,,,,,,,
Adzuna,1.0,2.0,3.0,0.0,6.0,2.0,3.0,6.0,12.0,0.0,...,6.0,27.0,0.0,11.0,0.0,2.0,22.0,2.0,4.0,0.0
Indeed,7.0,8.0,5.0,1.0,13.0,10.0,10.0,7.0,17.0,4.0,...,10.0,63.0,5.0,28.0,0.0,3.0,48.0,3.0,6.0,0.0
LinkedIn,6.0,4.0,0.0,0.0,1.0,7.0,14.0,2.0,20.0,1.0,...,6.0,61.0,1.0,24.0,4.0,1.0,27.0,0.0,4.0,1.0


### Estructuración de Skills por ofertas

In [7]:
# Identificar formato de skills
df_LinkedIn["skill"].apply(type).value_counts()



skill
<class 'str'>    97
Name: count, dtype: int64

In [8]:
# Cambio de formato a skills de string a lista
df_LinkedIn["skill"] = df_LinkedIn["skill"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_Indeed["skill"] = df_Indeed["skill"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_Adzuna["skill"] = df_Adzuna["skill"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)


In [9]:
df_LinkedIn_skills_ofertas = df_LinkedIn.explode("skill")
df_Indeed_skills_ofertas = df_Indeed.explode("skill")
df_Adzuna_skills_ofertas = df_Adzuna.explode("skill")

In [10]:
df_skills_ofertas = pd.concat([df_LinkedIn_skills_ofertas, df_Indeed_skills_ofertas, df_Adzuna_skills_ofertas], ignore_index=True)


### Estructuración de Familias

In [11]:
# Diccionario de skills y familias

DICCIONARIO_SKILLS = {
    "IDIOMAS" : {
        "español": ["español", "spanish", "castellano", "espanyol", "espagnol"],
        "inglés": ["inglés", "english", "inglés b2", "inglés c1", "inglés avanzado", "anglès", "anglais"],
        "catalán": ["catalán", "catalan", "català"],  
        "francés": ["francés", "french", "francès", "français"],
        "alemán": ["alemán", "german", "alemany" ],
        "italiano": ["italiano", "italian", "italià"],
        "portugués": ["portugués", "portuguese", "portugues"],
    },
    "SOFT_SKILLS" : {
        "comunicación": ["comunicación", "communication", "comunicarse", "comunicació", "communication"],
        "trabajo en equipo": ["trabajo en equipo", "teamwork", "colaboración", "collaboration", "treball en equip", "travail d'équipe"],
        "liderazgo": ["liderazgo", "leadership", "lideratge", "direction"],
        "pensamiento crítico": ["pensamiento crítico", "critical thinking", "pensament crític", "pensée critique"],
        "resolución de problemas": ["resolución de problemas", "problem solving", "resolució de problemes", "résolution de problèmes"],
        "organización": ["organización", "organizational", "organizado", "organització", "organisation"],
        "proactividad": ["proactividad", "proactive", "proactivitat", "proactivité"],
        "adaptabilidad": ["adaptabilidad", "adaptable", "adaptability", "adaptabilitat", "adaptabilité"], 
        "gestión del tiempo": ["gestión del tiempo", "time management", "gestió del temps", "gestion du temps"],
        "orientación a resultados": ["orientación a resultados", "results oriented", "orientació a resultats", "orientation vers les résultats"],
    },
    "HARD_SKILLS" : {
        "python": ["python", "py"],
        "r": [" r ", " r,", " r.", "lenguaje r"],
        "sql": ["sql", "postgres", "postgresql", "mysql", "bigquery"],
        "excel": ["excel", "microsoft excel"],
        "estadística": ["estadística", "statistics", "statistical", "statistiques"],
        "data analysis": ["data analysis", "análisis de datos", "anàlisi de dades", "analyse des données"],
        "data visualization": ["data visualization", "visualización de datos", "visualització de dades", "visualisation des données"],
        "etl": ["etl", "extract transform load"],
    },
    "BI_TOOLS" : {
        "power bi": ["power bi", "powerbi", "power-bi"],
        "tableau": ["tableau"],
        "looker studio": ["looker", "looker studio", "google data studio"],
        "qlik": ["qlik", "qlikview", "qliksense"],
    },
    "ML_AI" : {
        "machine learning": ["machine learning", "ml"],
        "deep learning": ["deep learning", "dl"],
        "scikit-learn": ["scikit-learn", "sklearn"],
        "tensorflow": ["tensorflow"],
        "pytorch": ["pytorch"],
        "nlp": ["nlp", "procesamiento de lenguaje natural"],
    },
    "DATA_ENGINEERING" : {
        "spark": ["spark", "pyspark"],
        "hadoop": ["hadoop"],
        "airflow": ["airflow"],
        "kafka": ["kafka"],
        "docker": ["docker"],
        "kubernetes": ["kubernetes", "k8s"],
        "git": ["git", "github", "control de versiones"],
    },
    "CLOUD" : {
        "aws": ["aws", "amazon web services"],
        "azure": ["azure","azure cloud"],
        "gcp": ["gcp", "google cloud"],
        "databricks": ["databricks"],
        "snowflake": ["snowflake"],
    },
    "FRAMEWORKS_LIBRERIAS" : {
        "pandas": ["pandas"],
        "numpy": ["numpy"],
        "scikit-learn": ["scikit-learn", "sklearn"],
        "tensorflow": ["tensorflow"],
        "pytorch": ["pytorch"],
        "matplotlib": ["matplotlib"],
        "seaborn": ["seaborn"],
        "keras": ["keras"],
        "xgboost": ["xgboost"],
        "lightgbm": ["lightgbm", "lgbm"],
        "statsmodels": ["statsmodels"],
    },
    "PROGRAMMING" : {
        "backend": ["backend", "back-end"],
        "frontend": ["frontend", "front-end"],
        "fullstack": ["fullstack", "full-stack"],
        "api": ["api", "rest api", "restful api"],
        "rest": ["rest", "restful"],
        "microservicios": ["microservicios", "microservices", "microserveis"],
        "php": ["php"],
        "java": ["java"],
        "javascript": ["javascript", "js"],
        "typescript": ["typescript", "ts"],
        "node": ["node", "nodejs", "node.js"],
        "react": ["react", "reactjs", "react.js"],
        "angular": ["angular"],
        "django": ["django"],
        "flask": ["flask"],
        "fastapi": ["fastapi"],
    },
    "DATA_ENGINEERING_EXTRA" : {
        "redshift": ["redshift", "amazon redshift"],
        "warehouse": ["data warehouse", "warehouse"],
        "datalake": ["data lake", "datalake"],
        "dbt": ["dbt", "data build tool"],
        "sql server": ["sql server", "mssql"],
        "oracle": ["oracle"],
        "mariadb": ["mariadb"],
        "mongodb": ["mongodb", "mongo"],
        "redis": ["redis"],
    },
    "DEVOPS" : {
        "devops": ["devops"],
        "ci/cd": ["ci/cd", "cicd", "continuous integration", "continuous delivery"],
        "terraform": ["terraform"],
        "ansible": ["ansible"],
    },
    "DATABASES_EXTRA" : {
        "bbdd": ["bbdd", "bases de datos", "base de dades", "dataset", "base de données"],
        "sql server": ["sql server", "mssql"],
        "oracle": ["oracle"],
        "mariadb": ["mariadb"],
        "mongodb": ["mongodb", "mongo"],
        "redis": ["redis"],
    }
}


In [12]:
# Crear el mapeo skill → familia
skill_a_familia = {}

for familia, skills in DICCIONARIO_SKILLS.items():
    for skill in skills:
        skill_a_familia[skill.lower()] = familia

# Asignar familia a cada skill
df_skills_ofertas["familia"] = df_skills_ofertas["skill"].map(skill_a_familia)


In [14]:
# Filtrar skills que pertenecen a una familia
df_familias = df_skills_ofertas.dropna(subset=["familia"])

# Agregar familias por portal
df_familias_conteo = (
    df_familias
    .groupby(["portal", "familia"])
    .size()
    .reset_index(name="conteo"))

# Generar tablas pivotadas y porcentuales
tab_familias, tab_familias_pct = generar_tablas_conteo(df_familias_conteo, "familia")


In [15]:
tab_familias

familia,BI_TOOLS,CLOUD,DATABASES_EXTRA,DATA_ENGINEERING,DATA_ENGINEERING_EXTRA,DEVOPS,FRAMEWORKS_LIBRERIAS,HARD_SKILLS,IDIOMAS,ML_AI,PROGRAMMING,SOFT_SKILLS
portal,,,,,,,,,,,,
Adzuna,35.0,12.0,13.0,31.0,18.0,6.0,0.0,97.0,39.0,9.0,41.0,90.0
Indeed,87.0,44.0,24.0,40.0,20.0,6.0,8.0,190.0,125.0,11.0,46.0,227.0
LinkedIn,93.0,37.0,23.0,22.0,9.0,5.0,27.0,203.0,73.0,26.0,7.0,128.0


### Estructuración de Requisitos

In [16]:
# Normalizar requisitos dentro de cada df_{portal}
cols_req = [
    "modalidad_nlp",
    "contrato_nlp",
    "jornada_nlp",
    "experiencia_nivel_nlp",
    "educacion_nlp"
]

for col in cols_req:
    df_LinkedIn[col] = (
        df_LinkedIn[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace("nan", None)
    )

for col in cols_req:
    df_Indeed[col] = (
        df_Indeed[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace("nan", None)
    )

for col in cols_req:
    df_Adzuna[col] = (
        df_Adzuna[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace("nan", None)
    )



In [17]:
df_LinkedIn["experiencia_años_nlp"] = pd.to_numeric(df_LinkedIn["experiencia_años_nlp"], errors="coerce")

df_LinkedIn["experiencia_bin"] = pd.cut(df_LinkedIn["experiencia_años_nlp"],
            bins=[0,1,3,5,10,50],labels=["0-1","1-3","3-5","5-10","+10"])


df_Indeed["experiencia_años_nlp"] = pd.to_numeric(df_Indeed["experiencia_años_nlp"], errors="coerce")

df_Indeed["experiencia_bin"] = pd.cut(df_Indeed["experiencia_años_nlp"],
            bins=[0,1,3,5,10,50],labels=["0-1","1-3","3-5","5-10","+10"])


df_Adzuna["experiencia_años_nlp"] = pd.to_numeric(df_Adzuna["experiencia_años_nlp"], errors="coerce")

df_Adzuna["experiencia_bin"] = pd.cut(df_Adzuna["experiencia_años_nlp"],
            bins=[0,1,3,5,10,50],labels=["0-1","1-3","3-5","5-10","+10"])

In [18]:
# Tablas modalidad
tab_modalidad_LKD, tab_modalidad_LKD_pct = generar_tablas(df_LinkedIn, "modalidad_nlp")
tab_modalidad_IND, tab_modalidad_IND_pct = generar_tablas(df_Indeed, "modalidad_nlp")
tab_modalidad_ADZ, tab_modalidad_ADZ_pct = generar_tablas(df_Adzuna, "modalidad_nlp")

In [19]:
# Tablas contrato
tab_contrato_LKD, tab_contrato_LKD_pct = generar_tablas(df_LinkedIn, "contrato_nlp")
tab_contrato_IND, tab_contrato_IND_pct = generar_tablas(df_Indeed, "contrato_nlp")
tab_contrato_ADZ, tab_contrato_ADZ_pct = generar_tablas(df_Adzuna, "contrato_nlp")

In [20]:
# Tablas jornada
tab_jornada_LKD, tab_jornada_LKD_pct = generar_tablas(df_LinkedIn, "jornada_nlp")
tab_jornada_IND, tab_jornada_IND_pct = generar_tablas(df_Indeed, "jornada_nlp")
tab_jornada_ADZ, tab_jornada_ADZ_pct = generar_tablas(df_Adzuna, "jornada_nlp")

In [21]:
# Tablas experiencia_nivel
tab_experiencia_nivel_LKD, tab_experiencia_nivel_LKD_pct = generar_tablas(df_LinkedIn, "experiencia_nivel_nlp")
tab_experiencia_nivel_IND, tab_experiencia_nivel_IND_pct = generar_tablas(df_Indeed, "experiencia_nivel_nlp")
tab_experiencia_nivel_ADZ, tab_experiencia_nivel_ADZ_pct = generar_tablas(df_Adzuna, "experiencia_nivel_nlp")

In [22]:
# Tablas educacion
tab_educacion_LKD, tab_educacion_LKD_pct = generar_tablas(df_LinkedIn, "educacion_nlp")
tab_educacion_IND, tab_educacion_IND_pct = generar_tablas(df_Indeed, "educacion_nlp")
tab_educacion_ADZ, tab_educacion_ADZ_pct = generar_tablas(df_Adzuna, "educacion_nlp")

In [23]:
# Tabla experiencia_bin
tab_experiencia_bin_LKD, tab_experiencia_bin_LKD_pct = generar_tablas(df_LinkedIn, "experiencia_bin")
tab_experiencia_bin_IND, tab_experiencia_bin_IND_pct = generar_tablas(df_Indeed, "experiencia_bin")
tab_experiencia_bin_ADZ, tab_experiencia_bin_ADZ_pct = generar_tablas(df_Adzuna, "experiencia_bin")

C:\Users\Usuario\AppData\Local\Temp\ipykernel_16276\491699686.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["portal", columna])
C:\Users\Usuario\AppData\Local\Temp\ipykernel_16276\491699686.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["portal", columna])
C:\Users\Usuario\AppData\Local\Temp\ipykernel_16276\491699686.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["portal", co

In [ ]:
# Concatenar DataFrames y crear DataFrame de requisitos
df_requisitos = pd.concat([df_LinkedIn, df_Indeed, df_Adzuna], ignore_index=True)

# Definir columnas de requisitos
requisitos = [
    "modalidad_nlp",
    "contrato_nlp",
    "jornada_nlp",
    "educacion_nlp",
    "experiencia_bin",
    "experiencia_nivel_nlp"
]

# Convertir a object (para permitir reemplazar NaN sin restricciones)
df_requisitos[requisitos] = df_requisitos[requisitos].astype("object")

# Reemplazar NaN por “N/A”
df_requisitos[requisitos] = df_requisitos[requisitos].fillna("N/A")

# Volver a convertir cada columna a Categorical (incluyendo “N/A”)
for col in requisitos:
    categorias = sorted(df_requisitos[col].unique())
    df_requisitos[col] = pd.Categorical(df_requisitos[col], categories=categorias)


In [29]:
# Generar tablas globales por requisito
tab_modalidad, tab_modalidad_pct = generar_tablas(df_requisitos, "modalidad_nlp")
tab_contrato, tab_contrato_pct = generar_tablas(df_requisitos, "contrato_nlp")
tab_jornada, tab_jornada_pct = generar_tablas(df_requisitos, "jornada_nlp")
tab_exp_nivel, tab_exp_nivel_pct = generar_tablas(df_requisitos, "experiencia_nivel_nlp")
tab_educacion, tab_educacion_pct = generar_tablas(df_requisitos, "educacion_nlp")
tab_exp_bin, tab_exp_bin_pct = generar_tablas(df_requisitos, "experiencia_bin")

C:\Users\Usuario\AppData\Local\Temp\ipykernel_16276\491699686.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["portal", columna])
C:\Users\Usuario\AppData\Local\Temp\ipykernel_16276\491699686.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["portal", columna])
C:\Users\Usuario\AppData\Local\Temp\ipykernel_16276\491699686.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["portal", co

### Estructuración de Tablas Cruzadas Familias↔Requisitos

In [45]:
# Preparar dataset cruzado
df_fam_req = df_familias.merge(
    df_requisitos[
        ["id_oferta", "portal", "modalidad_nlp", "contrato_nlp",
        "jornada_nlp", "educacion_nlp", "experiencia_bin", "experiencia_nivel_nlp"]],
    on=["id_oferta", "portal"],
    how="left"
)


In [ ]:
def limpiar_columna_duplicada(df, base_col):
    """
    Mantiene una sola columna cuando existen duplicados como base_col_x y base_col_y.
    Conserva la que tenga más valores no nulos.
    """
    # Detectar columnas relacionadas
    cols = [c for c in df.columns if c.startswith(base_col)]
    
    # Si solo hay una, no hay nada que limpiar
    if len(cols) == 1:
        df = df.rename(columns={cols[0]: base_col})
        return df
    
    # Elegir la columna con más valores válidos
    col_buena = max(cols, key=lambda c: df[c].notna().sum())
    
    # Renombrar la buena
    df = df.rename(columns={col_buena: base_col})
    
    # Eliminar las demás
    cols_malas = [c for c in cols if c != col_buena]
    df = df.drop(columns=cols_malas)
    
    return df


In [47]:
def generar_tablas_familia_requisito(df, requisito):
    """
    Genera:
    - tabla de conteos: familia × requisito
    - tabla porcentual por fila
    """
    # Tabla de conteos
    tabla = df.pivot_table(
        index="familia",
        columns=requisito,
        values="id_oferta",
        aggfunc="count",
        fill_value=0
    )
    
    # Tabla porcentual
    tabla_pct = tabla.div(tabla.sum(axis=1), axis=0)
    
    return tabla, tabla_pct


In [48]:
# Aplicar funcion limpiar_columna_duplicada
for req in requisitos:
    df_fam_req = limpiar_columna_duplicada(df_fam_req, req)


In [49]:
# Generar tablas familia × requisito
tablas_fam = {}
tablas_fam_pct = {}

for req in requisitos:
    tabla, tabla_pct = generar_tablas_familia_requisito(df_fam_req, req)
    tablas_fam[req] = tabla
    tablas_fam_pct[req] = tabla_pct


C:\Users\Usuario\AppData\Local\Temp\ipykernel_16276\3556473407.py:8: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  tabla = df.pivot_table(
C:\Users\Usuario\AppData\Local\Temp\ipykernel_16276\3556473407.py:8: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  tabla = df.pivot_table(
C:\Users\Usuario\AppData\Local\Temp\ipykernel_16276\3556473407.py:8: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  tabla = df.pivot_table(
C:\Users\Usuario\AppData\Local\Temp\ipykernel_16276\3556473407.py:8: FutureWarning: The def

### Exportar Datos

In [51]:
# Exportar archivo para visualizacion y analisis

# Skills por oferta
df_skills_ofertas.to_csv("df_skills_ofertas.csv", index=False)

# Skills globales
df_skills_global_conteo.to_csv("df_skills_global_conteo.csv", index=False)

# Familias por oferta
df_familias.to_csv("df_familias.csv", index=False)

# Familias agregadas
df_familias_conteo.to_csv("df_familias_conteo.csv", index=False)
tab_familias.to_csv("tab_familias.csv")
tab_familias_pct.to_csv("tab_familias_pct.csv")

# Requisitos por oferta
df_requisitos.to_csv("df_requisitos.csv", index=False)

# Tablas globales de requisitos
tab_modalidad.to_csv("tab_modalidad.csv")
tab_modalidad_pct.to_csv("tab_modalidad_pct.csv")

tab_contrato.to_csv("tab_contrato.csv")
tab_contrato_pct.to_csv("tab_contrato_pct.csv")

tab_jornada.to_csv("tab_jornada.csv")
tab_jornada_pct.to_csv("tab_jornada_pct.csv")

tab_exp_nivel.to_csv("tab_nivel_experiencia.csv")
tab_exp_nivel_pct.to_csv("tab_nivel_experiencia_pct.csv")

tab_educacion.to_csv("tab_educacion.csv")
tab_educacion_pct.to_csv("tab_educacion_pct.csv")

tab_exp_bin.to_csv("tab_experiencia_años.csv")
tab_exp_bin_pct.to_csv("tab_experiencia_años_pct.csv")

# Tablas cruzadas
df_fam_req.to_csv("df_fam_req.csv", index=False)

for req in requisitos:
    tablas_fam[req].to_csv(f"tabla_fam_{req}.csv")
    tablas_fam_pct[req].to_csv(f"tabla_fam_{req}_pct.csv")

